# File 04 LR — Lighting-Robust Segmented Preprocessing
Dataloaders and sanity checks. No training.


In [1]:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import EfficientNet_B0_Weights

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MANIFEST_PATH=Path(r'D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv')
OUTPUT_DIR=Path(r'D:\DIABETES\diabetes_pipeline_outputs\04_segmented_preprocessing_lighting_robust')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE=224; BATCH_SIZE=32; NUM_WORKERS=0; PIN_MEMORY=torch.cuda.is_available()
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
LABEL_MAP={'diabetes':1,'non_diabetes':0}
print(f'Device: {DEVICE}')


Device: cuda


## Lighting Correction


In [2]:
def correct_lighting_clahe(image_pil, clip_limit=1.5, tile_grid_size=(8,8)):
    img_rgb=np.array(image_pil.convert('RGB'))
    lab=cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq=clahe.apply(l)
    lab_eq=cv2.merge([l_eq,a,b])
    rgb_eq=cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)
    return Image.fromarray(rgb_eq)

def pad_square(img):
    w,h=img.size; s=max(w,h)
    new=Image.new('RGB',(s,s),(0,0,0))
    new.paste(img,((s-w)//2,(s-h)//2))
    return new

print('Lighting correction defined.')


Lighting correction defined.


## Load Manifest


In [3]:
df=pd.read_csv(MANIFEST_PATH)
df=df[df['export_status']=='ok'].reset_index(drop=True)
img_col='segmented_image_path' if 'segmented_image_path' in df.columns else 'crop_masked_path'
df['image_path']=df[img_col]
df['file_exists']=df['image_path'].apply(lambda p: Path(p).exists())
missing=(~df['file_exists']).sum()
if missing>0: print(f'WARNING: {missing} missing'); df=df[df['file_exists']].reset_index(drop=True)
for cls,exp in LABEL_MAP.items():
    obs=df[df['final_label']==cls]['label_binary'].unique()
    assert list(obs)==[exp], f'{cls} label mismatch'
print(f'Loaded {len(df)} rows. Label mapping PASS.')
df[['image_path','final_split','final_label','label_binary','file_exists']].to_csv(OUTPUT_DIR/'04_lr_manifest_check.csv',index=False)


Loaded 2750 rows. Label mapping PASS.


## Transforms and Datasets


In [4]:
train_transform=T.Compose([
    T.Lambda(correct_lighting_clahe),
    T.Lambda(pad_square),
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(7),
    T.RandomAffine(0,translate=(0.03,0.03),scale=(0.95,1.05)),
    T.ColorJitter(brightness=0.08,contrast=0.08,saturation=0.05,hue=0.01),
    T.ToTensor(),
    T.Normalize(mean=MEAN,std=STD),
])

val_test_transform=T.Compose([
    T.Lambda(correct_lighting_clahe),
    T.Lambda(pad_square),
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN,std=STD),
])

class SegDataset(Dataset):
    def __init__(self,df,transform):
        self.df=df.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=Image.open(row['image_path']).convert('RGB')
        if self.transform: img=self.transform(img)
        return img, int(row['label_binary'])

df_train=df[df['final_split']=='train']; df_val=df[df['final_split']=='val']; df_test=df[df['final_split']=='test']
ds_train=SegDataset(df_train,train_transform); ds_val=SegDataset(df_val,val_test_transform); ds_test=SegDataset(df_test,val_test_transform)
dl_train=DataLoader(ds_train,BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
dl_val=DataLoader(ds_val,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
dl_test=DataLoader(ds_test,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
print(f'Train:{len(ds_train)} Val:{len(ds_val)} Test:{len(ds_test)}')


Train:1930 Val:408 Test:412


## Sanity Checks and Outputs


In [5]:
checks=[]
for name,dl in [('train',dl_train),('val',dl_val),('test',dl_test)]:
    try:
        imgs,labels=next(iter(dl))
        checks.append({'split':name,'status':'PASS','shape':str(list(imgs.shape)),'labels':str(sorted(labels.unique().tolist())),'min':round(float(imgs.min()),4),'max':round(float(imgs.max()),4)})
        print(f'{name}: {list(imgs.shape)}, labels={sorted(labels.unique().tolist())}')
    except Exception as e:
        checks.append({'split':name,'status':'FAIL','notes':str(e)})
pd.DataFrame(checks).to_csv(OUTPUT_DIR/'04_lr_batch_sanity_check.csv',index=False)


train: [32, 3, 224, 224], labels=[0, 1]
val: [32, 3, 224, 224], labels=[1]
test: [32, 3, 224, 224], labels=[1]


In [6]:
# Before/after lighting correction visualization
sample_paths=df_train.sample(min(8,len(df_train)),random_state=SEED)['image_path'].tolist()
fig,axes=plt.subplots(2,len(sample_paths),figsize=(4*len(sample_paths),8))
for i,p in enumerate(sample_paths):
    orig=Image.open(p).convert('RGB')
    corr=correct_lighting_clahe(orig)
    axes[0,i].imshow(orig); axes[0,i].set_title('Original',fontsize=7); axes[0,i].axis('off')
    axes[1,i].imshow(corr); axes[1,i].set_title('CLAHE',fontsize=7); axes[1,i].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'04_lr_original_vs_lighting_corrected_examples.png',dpi=100)
plt.close()
print('Lighting comparison saved.')


Lighting comparison saved.


In [7]:
def denorm(t):
    t=t.clone()
    for c in range(3): t[c]=t[c]*STD[c]+MEAN[c]
    return t.clamp(0,1)
imgs,labels=next(iter(dl_train)); n=min(16,imgs.shape[0])
fig,axes=plt.subplots(4,4,figsize=(12,12))
for i in range(n):
    ax=axes[i//4,i%4]; ax.imshow(denorm(imgs[i]).permute(1,2,0).numpy())
    ax.set_title('D' if labels[i]==1 else 'ND',fontsize=8); ax.axis('off')
for i in range(n,16): axes[i//4,i%4].axis('off')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'04_lr_sample_batch_visualization.png',dpi=100); plt.close()

dl_rows=[]
for sp,shuf in [('train',True),('val',False),('test',False)]:
    sdf=df[df['final_split']==sp]
    dl_rows.append({'split':sp,'count':len(sdf),'diabetes':int((sdf['label_binary']==1).sum()),'non_diabetes':int((sdf['label_binary']==0).sum()),'batch_size':BATCH_SIZE,'shuffle':shuf,'device':str(DEVICE)})
pd.DataFrame(dl_rows).to_csv(OUTPUT_DIR/'04_lr_dataloader_summary.csv',index=False)

cm_rows=[]
for sp in ['train','val','test']:
    sdf=df[df['final_split']==sp]
    for cls,exp in LABEL_MAP.items():
        obs=sdf[sdf['final_label']==cls]['label_binary'].unique().tolist()
        cm_rows.append({'split':sp,'class':cls,'expected':exp,'observed':str(obs),'count':len(sdf[sdf['final_label']==cls]),'status':'PASS' if obs==[exp] else 'FAIL'})
pd.DataFrame(cm_rows).to_csv(OUTPUT_DIR/'04_lr_class_mapping_verification.csv',index=False)
pd.DataFrame([{'total':len(df),'missing':missing,'status':'PASS' if missing==0 else 'FAIL'}]).to_csv(OUTPUT_DIR/'04_lr_file_existence_check.csv',index=False)

txt='FILE 04 LR TRANSFORMS\nLighting: CLAHE LAB L-channel (clip=1.5, tile=8x8)\nTrain: CLAHE→pad→resize→HFlip→Rot(7°)→Affine→ColorJitter→ToTensor→Normalize\nVal/Test: CLAHE→pad→resize→ToTensor→Normalize (deterministic)\n'
with open(OUTPUT_DIR/'04_lr_transform_summary.txt','w', encoding='utf-8') as f: f.write(txt)

handoff=f'FILE 04 LR HANDOFF\nStatus: PASS\nManifest: {MANIFEST_PATH}\nTrain:{len(df_train)} Val:{len(df_val)} Test:{len(df_test)}\nLighting: CLAHE LAB L-channel\nBatch: {BATCH_SIZE} Workers: {NUM_WORKERS} Device: {DEVICE}\nBatch sanity: PASS\n'
with open(OUTPUT_DIR/'04_lr_handoff_summary.txt','w', encoding='utf-8') as f: f.write(handoff)
print(handoff)


FILE 04 LR HANDOFF
Status: PASS
Manifest: D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv
Train:1930 Val:408 Test:412
Lighting: CLAHE LAB L-channel
Batch: 32 Workers: 0 Device: cuda
Batch sanity: PASS

